# US E-Commerce Analytics with PySpark (Modules 1–5 Combined)

This notebook combines five Spark course notebooks (originally built on the Olist dataset) into one, using the **US E-Commerce** dataset stored in HDFS at `/data/us-ecom/`.

**Data used (7 files):** `orders.csv`, `order_items.csv`, `order_payments.csv`, `order_reviews.csv`, `products.csv`, `sellers.csv`, `geolocation.csv`

**Changes from the Olist version**
- No `customers.csv` and no category-translation file, so customer-table cells (customer state, customer zip, customer de-duplication) were removed.
- Geolocation is joined on the **seller** zip code (`seller_zip_code_prefix` → `zip_code_prefix`).
- `products.csv` has its own `price` column (base retail price), renamed to `base_price` so it doesn't clash with `order_items.price`.
- Payment type labels updated for US values (paypal, apple_pay, bank_transfer, …).

| Module | Topic |
|---|---|
| 1 | Data Ingestion and Exploration |
| 2 | Data Cleaning and Transformation |
| 3 | Data Integration and Aggregation |
| 4 | Performance Optimization |
| 5 | Data Serving |

# Module 1 – Data Ingestion and Exploration

### Step 1: Setting Up the Spark Environment

In a real-world company setup we would:
1. **Deploy a Spark cluster** (AWS EMR, GCP Dataproc, Azure HDInsight, or on-prem Hadoop).
2. **Store data in HDFS** instead of local storage.
3. **Use PySpark** to interact with the data.

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
.appName('USEcomData') \
.config('spark.executor.memory', '2g') \
.config('spark.executor.cores', '1') \
.getOrCreate()

spark.range(10).count()   # quick test: should print 10 within a few seconds

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/01 20:36:20 INFO SparkEnv: Registering MapOutputTracker
26/11/01 20:36:20 INFO SparkEnv: Registering BlockManagerMaster
26/11/01 20:36:20 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/11/01 20:36:20 INFO SparkEnv: Registering OutputCommitCoordinator


10

In [2]:
spark

In [3]:
!hadoop fs -ls /data/us-ecom/

Found 8 items
-rw-r--r--   2 spate664 hadoop  120276557 2026-11-01 01:20 /data/us-ecom/customers.csv
-rw-r--r--   2 spate664 hadoop     638247 2026-11-01 01:20 /data/us-ecom/geolocation.csv
-rw-r--r--   2 spate664 hadoop  330261636 2026-11-01 01:20 /data/us-ecom/order_items.csv
-rw-r--r--   2 spate664 hadoop   67791945 2026-11-01 01:20 /data/us-ecom/order_payments.csv
-rw-r--r--   2 spate664 hadoop  157110590 2026-11-01 01:20 /data/us-ecom/order_reviews.csv
-rw-r--r--   2 spate664 hadoop  181416338 2026-11-01 01:20 /data/us-ecom/orders.csv
-rw-r--r--   2 spate664 hadoop     204730 2026-11-01 01:20 /data/us-ecom/products.csv
-rw-r--r--   2 spate664 hadoop      46352 2026-11-01 01:20 /data/us-ecom/sellers.csv


In [4]:
hdfs_path = '/data/us-ecom/'

orders_df      = spark.read.csv(hdfs_path + 'orders.csv',         header=True, inferSchema=True)
order_item_df  = spark.read.csv(hdfs_path + 'order_items.csv',    header=True, inferSchema=True)
payments_df    = spark.read.csv(hdfs_path + 'order_payments.csv', header=True, inferSchema=True)
reviews_df     = spark.read.csv(hdfs_path + 'order_reviews.csv',  header=True, inferSchema=True, multiLine=True, escape='"')
products_df    = spark.read.csv(hdfs_path + 'products.csv',       header=True, inferSchema=True)
sellers_df     = spark.read.csv(hdfs_path + 'sellers.csv',        header=True, inferSchema=True)
geolocation_df = spark.read.csv(hdfs_path + 'geolocation.csv',    header=True, inferSchema=True)

# products.csv has a 'price' column (base retail price) and order_items.csv has 'price' (final selling price).
# Rename the product one so later joins don't end up with two 'price' columns.
products_df = products_df.withColumnRenamed('price', 'base_price')

In [5]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [6]:
print(f'Orders : {orders_df.count()} rows')
print(f'Order items : {order_item_df.count()} rows')

Orders : 1000000 rows
Order items : 2199819 rows


In [7]:
orders_df.columns

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

In [8]:
from pyspark.sql.functions import col

# Check for nulls in critical fields
orders_df.select([col(c).isNull().alias(c) for c in orders_df.columns]).show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|   false|      false|       false|                   false|            false|                        true|                         true|                        false|
|   false|      false|       false|                   false|            false|                       false|                        false|                        false|
|   false|      false|       false|                   false|            false|                       false|                        false|                       

In [9]:
from pyspark.sql.functions import col,when ,count

# Check for nulls in critical fields
orders_df.select([count(when(col(c).isNull(),1)).alias(c) for c in orders_df.columns]).show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|                0|                       66252|                        66252|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [10]:
# Duplicate Values

orders_df.groupBy('order_id').count().filter('count>1').show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [11]:
orders_df.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|720fd9e9-cdeb-4de...|1e2e2881-a0eb-4cb...|    canceled|     2025-12-27 07:07:20|2025-12-27 08:33:20|                        NULL|                         NULL|          2026-01-04 08:33:20|
|c0142972-63fa-4af...|380b7418-308c-4bf...|   delivered|     2019-06-07 19:30:44|2019-06-08 05:08:44|         2019-06-09 05:08:44|          2019-06-14 05:08:44|          2019-06-16 05:08:44|
|11bdf634-2b87-4d3...|89b0d980-868f-478...|  

In [12]:
# Order - Order status distribution

orders_df.groupBy('order_status').count().orderBy('count',ascending=False).show()

+------------+------+
|order_status| count|
+------------+------+
|   delivered|933748|
|    canceled| 66252|
+------------+------+



In [13]:
# Payments

payments_df.show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|720fd9e9-cdeb-4de...|                 1|      paypal|                   1|      2084.01|
|c0142972-63fa-4af...|                 1| credit_card|                   6|      1406.07|
|11bdf634-2b87-4d3...|                 1|     voucher|                   1|        911.3|
|11bdf634-2b87-4d3...|                 2| credit_card|                   8|       600.57|
|d29b58ad-af2a-47b...|                 1| credit_card|                  11|       170.73|
|d29b58ad-af2a-47b...|                 2|     voucher|                   1|       155.34|
|53fabbc8-fd4f-4c1...|                 1|     voucher|                   1|      2229.36|
|6ee6f6b0-f7fa-43b...|                 1| credit_card|                  11|      2120.33|
|79c3bd90-

In [14]:
payments_df.groupBy('payment_type').count().orderBy('count',ascending=False).show()

+-------------+------+
| payment_type| count|
+-------------+------+
|  credit_card|536416|
|       paypal|181278|
|      voucher|123150|
|bank_transfer| 96766|
|    apple_pay| 92114|
|   debit_card| 91034|
|       boleto| 28613|
+-------------+------+



In [15]:
#Top selling Products

In [16]:
order_item_df.show()

+--------------------+-------------+--------------------+--------------------+-------------------+-------+-------------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|  price|freight_value|discount_rate|
+--------------------+-------------+--------------------+--------------------+-------------------+-------+-------------+-------------+
|720fd9e9-cdeb-4de...|            1|89804d82-33a1-455...|36c2b043-ccf4-4d9...|2025-12-30 07:07:20|  916.9|        45.47|          0.0|
|720fd9e9-cdeb-4de...|            2|ac10b78c-342e-4b5...|09936504-415b-4f9...|2025-12-30 07:07:20| 817.14|       172.48|          0.0|
|720fd9e9-cdeb-4de...|            3|f18e6310-0e75-4b7...|7d336012-7101-4a6...|2025-12-30 07:07:20|  37.04|        94.98|          0.0|
|c0142972-63fa-4af...|            1|5816f107-b725-4c4...|91fe2cc8-51a5-4c7...|2019-06-10 19:30:44| 917.98|        27.42|          0.0|
|c0142972-63fa-4af...|            2|f4b3ce27-568d-4c3..

In [17]:
from pyspark.sql.functions import sum

top_products  = order_item_df.groupBy('product_id').agg(sum('price').alias('total_sales'))
top_products.orderBy('total_sales',ascending=False).show(20)

+--------------------+------------------+
|          product_id|       total_sales|
+--------------------+------------------+
|83fbdf52-21a3-4b6...| 4938901.800000058|
|9086b2b2-6100-410...| 4894925.350000073|
|d27b9b19-b9a9-4ea...| 4893846.909999908|
|7943b8ca-e76d-41b...| 4877712.600000091|
|0be8ed49-fc0e-484...| 4836486.720000056|
|cbceed2a-2fef-4be...| 4823852.339999946|
|3952c856-34d0-435...| 4823155.260000035|
|0726d086-56b4-4a9...| 4761642.850000049|
|8a1b95bf-694e-44a...|4760142.6599999145|
|b19d2bb6-43a7-4ef...| 4758172.379999967|
|95439579-e05b-41f...| 4745960.790000031|
|e7fbc23d-039d-41e...| 4692055.199999918|
|d3dd31c3-b005-477...|  4668562.15999997|
|c9b2a745-081b-4fc...| 4660957.759999918|
|67c471fe-5daa-4ce...| 4660705.739999943|
|40f21514-fa42-466...| 4628206.390000056|
|ee73dffc-1123-47d...|  4569926.39999995|
|28fbedaf-30dc-4ba...| 4538494.990000054|
|b0013bdd-ccec-49b...| 4493991.150000064|
|1e3e6aa5-0098-4f6...|        4472192.25|
+--------------------+------------

In [18]:
# Average Delivery Time Analysis 

delivery_df = orders_df.select('order_id','order_purchase_timestamp','order_delivered_customer_date')

In [19]:
delivery_df.show()

+--------------------+------------------------+-----------------------------+
|            order_id|order_purchase_timestamp|order_delivered_customer_date|
+--------------------+------------------------+-----------------------------+
|720fd9e9-cdeb-4de...|     2025-12-27 07:07:20|                         NULL|
|c0142972-63fa-4af...|     2019-06-07 19:30:44|          2019-06-14 05:08:44|
|11bdf634-2b87-4d3...|     2023-04-02 14:39:33|          2023-04-08 23:42:33|
|d29b58ad-af2a-47b...|     2023-01-28 12:09:24|          2023-02-02 15:26:24|
|53fabbc8-fd4f-4c1...|     2019-06-05 22:33:47|                         NULL|
|6ee6f6b0-f7fa-43b...|     2022-01-08 13:45:24|                         NULL|
|79c3bd90-97f4-49a...|     2025-12-26 22:21:53|                         NULL|
|3d4a035e-588c-4eb...|     2022-01-20 17:39:58|          2022-01-23 22:36:58|
|d7537ba2-da79-4a2...|     2021-12-04 09:44:26|          2021-12-14 19:09:26|
|e6441daf-f23c-4a2...|     2025-10-08 10:04:45|          2025-10

In [20]:
from pyspark.sql.functions import datediff,to_date

delivery_detail_df = delivery_df.withColumn('delivery_time',datediff(col('order_delivered_customer_date'),col('order_purchase_timestamp')))
                                    
    
delivery_detail_df.show()

+--------------------+------------------------+-----------------------------+-------------+
|            order_id|order_purchase_timestamp|order_delivered_customer_date|delivery_time|
+--------------------+------------------------+-----------------------------+-------------+
|720fd9e9-cdeb-4de...|     2025-12-27 07:07:20|                         NULL|         NULL|
|c0142972-63fa-4af...|     2019-06-07 19:30:44|          2019-06-14 05:08:44|            7|
|11bdf634-2b87-4d3...|     2023-04-02 14:39:33|          2023-04-08 23:42:33|            6|
|d29b58ad-af2a-47b...|     2023-01-28 12:09:24|          2023-02-02 15:26:24|            5|
|53fabbc8-fd4f-4c1...|     2019-06-05 22:33:47|                         NULL|         NULL|
|6ee6f6b0-f7fa-43b...|     2022-01-08 13:45:24|                         NULL|         NULL|
|79c3bd90-97f4-49a...|     2025-12-26 22:21:53|                         NULL|         NULL|
|3d4a035e-588c-4eb...|     2022-01-20 17:39:58|          2022-01-23 22:36:58|   

In [21]:
delivery_detail_df.orderBy('delivery_time',ascending=False).show()

+--------------------+------------------------+-----------------------------+-------------+
|            order_id|order_purchase_timestamp|order_delivered_customer_date|delivery_time|
+--------------------+------------------------+-----------------------------+-------------+
|7720fb5b-d74f-46f...|     2025-03-09 21:07:03|          2025-04-05 03:31:03|           27|
|fa1140b0-2076-4e5...|     2023-07-08 19:28:46|          2023-08-04 02:56:46|           27|
|19e5eba4-0ef3-4d1...|     2023-02-25 22:27:02|          2023-03-24 03:17:02|           27|
|5b026a31-64d0-4a0...|     2025-05-01 21:40:12|          2025-05-28 05:02:12|           27|
|26585d0b-23fa-4ad...|     2019-11-10 22:54:59|          2019-12-07 03:34:59|           27|
|e73ab926-e2c9-497...|     2019-12-13 19:42:27|          2020-01-09 04:14:27|           27|
|e526ec58-faaa-4dd...|     2024-07-26 22:16:50|          2024-08-22 00:24:50|           27|
|3e511970-9317-4ba...|     2019-10-23 18:23:54|          2019-11-19 00:44:54|   

In [22]:
delivery_detail_df

DataFrame[order_id: string, order_purchase_timestamp: timestamp, order_delivered_customer_date: timestamp, delivery_time: int]

# Module 2 - Data Cleaning and Transformation

In [23]:
# Identify Missing Values

In [24]:
from pyspark.sql.functions import *

In [25]:
def missing_values(df,df_name):
    print(f'Missing values in {df_name}:')
    df.select([count(when(col(c).isNull(),1)).alias(c) for c in df.columns]).show()

In [26]:
missing_values(orders_df,'orders')

Missing values in orders:


+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|                0|                       66252|                        66252|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [27]:
missing_values(order_item_df,'order_item')

Missing values in order_item:


+--------+-------------+----------+---------+-------------------+-----+-------------+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|discount_rate|
+--------+-------------+----------+---------+-------------------+-----+-------------+-------------+
|       0|            0|         0|        0|                  0|    0|            0|            0|
+--------+-------------+----------+---------+-------------------+-----+-------------+-------------+



In [28]:
missing_values(payments_df,'payments')

Missing values in payments:


+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|       0|                 0|           0|                   0|            0|
+--------+------------------+------------+--------------------+-------------+



# Handle Missing Values

1. Drop missing Values ( for non - critical columns )

2. Fill missing values ( for numerical columns )

3. Impute Missing Values ( for continous data )

In [29]:
orders_df_cleaned = orders_df.na.drop(subset=['order_id','customer_id','order_status'])

In [30]:
orders_df_cleaned.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|720fd9e9-cdeb-4de...|1e2e2881-a0eb-4cb...|    canceled|     2025-12-27 07:07:20|2025-12-27 08:33:20|                        NULL|                         NULL|          2026-01-04 08:33:20|
|c0142972-63fa-4af...|380b7418-308c-4bf...|   delivered|     2019-06-07 19:30:44|2019-06-08 05:08:44|         2019-06-09 05:08:44|          2019-06-14 05:08:44|          2019-06-16 05:08:44|
|11bdf634-2b87-4d3...|89b0d980-868f-478...|  

In [31]:
orders_df_cleaned = orders_df.fillna({'order_delivered_customer_date':'9999-12-31'})

In [32]:
orders_df_cleaned.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|720fd9e9-cdeb-4de...|1e2e2881-a0eb-4cb...|    canceled|     2025-12-27 07:07:20|2025-12-27 08:33:20|                        NULL|          9999-12-31 00:00:00|          2026-01-04 08:33:20|
|c0142972-63fa-4af...|380b7418-308c-4bf...|   delivered|     2019-06-07 19:30:44|2019-06-08 05:08:44|         2019-06-09 05:08:44|          2019-06-14 05:08:44|          2019-06-16 05:08:44|
|11bdf634-2b87-4d3...|89b0d980-868f-478...|  

# Impute missing values 

In [33]:
from pyspark.ml.feature import Imputer

imputer = Imputer(inputCols=['payment_value'],outputCols=['payment_value_imputed']).setStrategy('mean')


payments_df_cleaned = imputer.fit(payments_df).transform(payments_df)

In [34]:
payments_df_cleaned.show()

+--------------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+------------+--------------------+-------------+---------------------+
|720fd9e9-cdeb-4de...|                 1|      paypal|                   1|      2084.01|              2084.01|
|c0142972-63fa-4af...|                 1| credit_card|                   6|      1406.07|              1406.07|
|11bdf634-2b87-4d3...|                 1|     voucher|                   1|        911.3|                911.3|
|11bdf634-2b87-4d3...|                 2| credit_card|                   8|       600.57|               600.57|
|d29b58ad-af2a-47b...|                 1| credit_card|                  11|       170.73|               170.73|
|d29b58ad-af2a-47b...|                 2|     voucher|                   1|       155.34|               

In [35]:
payments_df_with_null = payments_df.withColumn('payment_value',when(col('payment_value')!= 99.33,col('payment_value')).otherwise(lit(None)))

In [36]:
payments_df_with_null.show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|720fd9e9-cdeb-4de...|                 1|      paypal|                   1|      2084.01|
|c0142972-63fa-4af...|                 1| credit_card|                   6|      1406.07|
|11bdf634-2b87-4d3...|                 1|     voucher|                   1|        911.3|
|11bdf634-2b87-4d3...|                 2| credit_card|                   8|       600.57|
|d29b58ad-af2a-47b...|                 1| credit_card|                  11|       170.73|
|d29b58ad-af2a-47b...|                 2|     voucher|                   1|       155.34|
|53fabbc8-fd4f-4c1...|                 1|     voucher|                   1|      2229.36|
|6ee6f6b0-f7fa-43b...|                 1| credit_card|                  11|      2120.33|
|79c3bd90-

In [37]:
from pyspark.ml.feature import Imputer

imputer = Imputer(inputCols=['payment_value'],outputCols=['payment_value_imputed']).setStrategy('median')


payments_df_cleaned = imputer.fit(payments_df_with_null).transform(payments_df_with_null)

payments_df_cleaned.show()

+--------------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+------------+--------------------+-------------+---------------------+
|720fd9e9-cdeb-4de...|                 1|      paypal|                   1|      2084.01|              2084.01|
|c0142972-63fa-4af...|                 1| credit_card|                   6|      1406.07|              1406.07|
|11bdf634-2b87-4d3...|                 1|     voucher|                   1|        911.3|                911.3|
|11bdf634-2b87-4d3...|                 2| credit_card|                   8|       600.57|               600.57|
|d29b58ad-af2a-47b...|                 1| credit_card|                  11|       170.73|               170.73|
|d29b58ad-af2a-47b...|                 2|     voucher|                   1|       155.34|               

# Standardizing the format

In [38]:
def print_schema(df,df_name):
    print(f'schema of {df_name}:')
    df.printSchema()

In [39]:
print_schema(orders_df,'orders')

schema of orders:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [40]:
print_schema(payments_df,'payments')

schema of payments:
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



In [41]:
orders_df.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|720fd9e9-cdeb-4de...|1e2e2881-a0eb-4cb...|    canceled|     2025-12-27 07:07:20|2025-12-27 08:33:20|                        NULL|                         NULL|          2026-01-04 08:33:20|
|c0142972-63fa-4af...|380b7418-308c-4bf...|   delivered|     2019-06-07 19:30:44|2019-06-08 05:08:44|         2019-06-09 05:08:44|          2019-06-14 05:08:44|          2019-06-16 05:08:44|
|11bdf634-2b87-4d3...|89b0d980-868f-478...|  

In [42]:
orders_df_cleaned = orders_df_cleaned.withColumn('order_purchase_timestamp',to_date(col('order_purchase_timestamp')))

In [43]:
orders_df_cleaned.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|720fd9e9-cdeb-4de...|1e2e2881-a0eb-4cb...|    canceled|              2025-12-27|2025-12-27 08:33:20|                        NULL|          9999-12-31 00:00:00|          2026-01-04 08:33:20|
|c0142972-63fa-4af...|380b7418-308c-4bf...|   delivered|              2019-06-07|2019-06-08 05:08:44|         2019-06-09 05:08:44|          2019-06-14 05:08:44|          2019-06-16 05:08:44|
|11bdf634-2b87-4d3...|89b0d980-868f-478...|  

In [44]:
payments_df_cleaned.show()

+--------------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+------------+--------------------+-------------+---------------------+
|720fd9e9-cdeb-4de...|                 1|      paypal|                   1|      2084.01|              2084.01|
|c0142972-63fa-4af...|                 1| credit_card|                   6|      1406.07|              1406.07|
|11bdf634-2b87-4d3...|                 1|     voucher|                   1|        911.3|                911.3|
|11bdf634-2b87-4d3...|                 2| credit_card|                   8|       600.57|               600.57|
|d29b58ad-af2a-47b...|                 1| credit_card|                  11|       170.73|               170.73|
|d29b58ad-af2a-47b...|                 2|     voucher|                   1|       155.34|               

In [45]:
payments_df_cleaned = payments_df_cleaned.withColumn('payment_type',when(col('payment_type')=='credit_card','Credit Card')
                                                     .when(col('payment_type')=='debit_card','Debit Card')
                                                     .when(col('payment_type')=='bank_transfer','Bank Transfer')
                                                     .when(col('payment_type')=='boleto','Boleto')
                                                     .when(col('payment_type')=='paypal','PayPal')
                                                     .when(col('payment_type')=='apple_pay','Apple Pay')
                                                     .when(col('payment_type')=='voucher','Voucher')
                                                    .otherwise('Other'))

In [46]:
payments_df_cleaned.show()

+--------------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+------------+--------------------+-------------+---------------------+
|720fd9e9-cdeb-4de...|                 1|      PayPal|                   1|      2084.01|              2084.01|
|c0142972-63fa-4af...|                 1| Credit Card|                   6|      1406.07|              1406.07|
|11bdf634-2b87-4d3...|                 1|     Voucher|                   1|        911.3|                911.3|
|11bdf634-2b87-4d3...|                 2| Credit Card|                   8|       600.57|               600.57|
|d29b58ad-af2a-47b...|                 1| Credit Card|                  11|       170.73|               170.73|
|d29b58ad-af2a-47b...|                 2|     Voucher|                   1|       155.34|               

# Remove Duplicate Records

In [47]:
orders_df_cleaned = orders_df_cleaned.dropDuplicates(['order_id'])

# Data Transformation

In [48]:
order_with_details = orders_df_cleaned.join(order_item_df,'order_id','left')\
.join(payments_df_cleaned,'order_id','left')

In [49]:
order_with_details.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+--------------------+--------------------+-------------------+-------+-------------+-------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|order_item_id|          product_id|           seller_id|shipping_limit_date|  price|freight_value|discount_rate|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+------------

In [50]:
order_with_total_value = order_with_details.groupBy('order_id')\
.agg(sum('payment_value').alias('total_order_value'))

In [51]:
order_with_total_value.show(5)

+--------------------+-----------------+
|            order_id|total_order_value|
+--------------------+-----------------+
|fd3aa30f-c604-49a...|          1309.12|
|6a98a56b-509f-419...|          9888.81|
|7541fdac-b670-41c...|          3794.76|
|24ace551-4a09-40e...|            971.6|
|3c5a809e-4743-425...|           8684.7|
+--------------------+-----------------+
only showing top 5 rows



# Advance Transformation 

In [52]:
order_item_df.show()

+--------------------+-------------+--------------------+--------------------+-------------------+-------+-------------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|  price|freight_value|discount_rate|
+--------------------+-------------+--------------------+--------------------+-------------------+-------+-------------+-------------+
|720fd9e9-cdeb-4de...|            1|89804d82-33a1-455...|36c2b043-ccf4-4d9...|2025-12-30 07:07:20|  916.9|        45.47|          0.0|
|720fd9e9-cdeb-4de...|            2|ac10b78c-342e-4b5...|09936504-415b-4f9...|2025-12-30 07:07:20| 817.14|       172.48|          0.0|
|720fd9e9-cdeb-4de...|            3|f18e6310-0e75-4b7...|7d336012-7101-4a6...|2025-12-30 07:07:20|  37.04|        94.98|          0.0|
|c0142972-63fa-4af...|            1|5816f107-b725-4c4...|91fe2cc8-51a5-4c7...|2019-06-10 19:30:44| 917.98|        27.42|          0.0|
|c0142972-63fa-4af...|            2|f4b3ce27-568d-4c3..

In [53]:
quantiles = order_item_df.approxQuantile('price',[0.01,0.99],0.01)
low_cutoff,high_cutoff = quantiles[0],quantiles[1]

In [54]:
low_cutoff,high_cutoff

(10.01, 1999.85)

In [55]:
order_item_df_cleaned = order_item_df.filter((col('price') >=low_cutoff) & (col('price') <=high_cutoff))

In [56]:
payments_df_cleaned.select('payment_installments').summary().show()

+-------+--------------------+
|summary|payment_installments|
+-------+--------------------+
|  count|             1149371|
|   mean|  3.9498499614136775|
| stddev|  3.7325479591682087|
|    min|                   1|
|    25%|                   1|
|    50%|                   1|
|    75%|                   7|
|    max|                  12|
+-------+--------------------+



In [57]:
products_df.show()

+--------------------+---------------------+--------------------+-------------+----------------+-----------------+-----------------+----------------+-------+----------+
|          product_id|product_category_name|        product_name|product_brand|product_weight_g|product_length_cm|product_height_cm|product_width_cm|   cost|base_price|
+--------------------+---------------------+--------------------+-------------+----------------+-----------------+-----------------+----------------+-------+----------+
|cdc4381f-fa67-41f...|                books|When successful m...|       Vertex|             661|               18|               19|               9|  22.45|     42.42|
|1f0d68f8-18d7-47c...|                 auto|Pulse Seat Cover ...|        Pulse|            9662|               52|               12|              25| 124.84|    148.16|
|f68621b0-8316-4cc...|          electronics|Nova M247 Headphones|         Nova|            4287|               12|               18|              15| 704.2

In [58]:
products_df_cleaned = products_df.withColumn(
    'product_size_category',
    when(col('product_weight_g') <500,'Small')
    .when(col('product_weight_g').between(500,2000),'Medium')
    .otherwise('Large')
)

In [59]:
products_df_cleaned.show()

+--------------------+---------------------+--------------------+-------------+----------------+-----------------+-----------------+----------------+-------+----------+---------------------+
|          product_id|product_category_name|        product_name|product_brand|product_weight_g|product_length_cm|product_height_cm|product_width_cm|   cost|base_price|product_size_category|
+--------------------+---------------------+--------------------+-------------+----------------+-----------------+-----------------+----------------+-------+----------+---------------------+
|cdc4381f-fa67-41f...|                books|When successful m...|       Vertex|             661|               18|               19|               9|  22.45|     42.42|               Medium|
|1f0d68f8-18d7-47c...|                 auto|Pulse Seat Cover ...|        Pulse|            9662|               52|               12|              25| 124.84|    148.16|                Large|
|f68621b0-8316-4cc...|          electronics|N

In [60]:
!hadoop fs -mkdir -p /data/us-ecom-proc

In [61]:
order_with_details.write.mode('overwrite').parquet('/data/us-ecom-proc/cleaned_data.parquet')

In [62]:
products_df_cleaned.write.mode('overwrite').parquet('/data/us-ecom-proc/product_df_cleaned.parquet')

In [63]:
!hadoop fs -ls /data/us-ecom-proc

Found 2 items
drwxr-xr-x   - root hadoop          0 2026-11-01 20:38 /data/us-ecom-proc/cleaned_data.parquet
drwxr-xr-x   - root hadoop          0 2026-11-01 20:38 /data/us-ecom-proc/product_df_cleaned.parquet


In [64]:
order_with_details.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: date (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- discount_rate: double (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- payment_value_imputed: double (nullable = true)



In [65]:
products_df_cleaned.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_brand: string (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- cost: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- product_size_category: string (nullable = false)



In [66]:
# Register the cleaned products Parquet as an external table (schema is read from the Parquet files)
spark.sql('DROP TABLE IF EXISTS cleaned_products')
spark.catalog.createTable('cleaned_products', path='/data/us-ecom-proc/product_df_cleaned.parquet', source='parquet')
spark.sql('SELECT * FROM cleaned_products LIMIT 5').show()

ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used
26/11/01 20:38:51 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


+--------------------+---------------------+--------------------+-------------+----------------+-----------------+-----------------+----------------+------+----------+---------------------+
|          product_id|product_category_name|        product_name|product_brand|product_weight_g|product_length_cm|product_height_cm|product_width_cm|  cost|base_price|product_size_category|
+--------------------+---------------------+--------------------+-------------+----------------+-----------------+-----------------+----------------+------+----------+---------------------+
|cdc4381f-fa67-41f...|                books|When successful m...|       Vertex|             661|               18|               19|               9| 22.45|     42.42|               Medium|
|1f0d68f8-18d7-47c...|                 auto|Pulse Seat Cover ...|        Pulse|            9662|               52|               12|              25|124.84|    148.16|                Large|
|f68621b0-8316-4cc...|          electronics|Nova M

# Module 3 - Data Integration and Aggregation

In [67]:
# Cache Frequently used Data for Better Performance

orders_df.cache()
order_item_df.cache()

26/11/01 20:38:52 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.
26/11/01 20:38:52 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.


DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double, discount_rate: double]

In [68]:
orders_items_joined_df = orders_df.join(order_item_df,'order_id','inner')

In [69]:
orders_items_products_df = orders_items_joined_df.join(products_df,'product_id','inner')

In [70]:
orders_items_products_sellers_df = orders_items_products_df.join(sellers_df,'seller_id','inner')

In [71]:
full_orders_df = orders_items_products_sellers_df   # no customers.csv in US E-Commerce, so no customer join

In [72]:
# Geolocation Data
# No customer zip code is available, so geolocation is joined on the SELLER zip code.
# geolocation.csv can have several rows per zip prefix -> keep one per zip so order rows aren't duplicated.
geolocation_unique_df = geolocation_df.dropDuplicates(['zip_code_prefix'])

full_orders_df = full_orders_df.join(geolocation_unique_df,full_orders_df.seller_zip_code_prefix == geolocation_unique_df.zip_code_prefix,'left')

In [73]:
full_orders_df = full_orders_df.join(reviews_df,'order_id','left')

In [74]:
full_orders_df = full_orders_df.join(payments_df,'order_id','left')

In [75]:
full_orders_df.cache()

26/11/01 20:38:53 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.
26/11/01 20:38:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[order_id: string, seller_id: string, product_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, discount_rate: double, product_category_name: string, product_name: string, product_brand: string, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, cost: double, base_price: double, seller_company_name: string, seller_contact_name: string, seller_contact_gender: string, seller_contact_age: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: int, review_comment_title: string, re

In [76]:
from pyspark.sql.functions import *

In [77]:
full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- discount_rate: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_brand: string (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_he

In [78]:
# Total Revenues Per Seller


seller_revenue_df = full_orders_df.groupBy('seller_id').agg(sum('price'))

In [79]:
seller_revenue_df.show(5)

+--------------------+------------------+
|           seller_id|        sum(price)|
+--------------------+------------------+
|d65674c9-cdb9-47f...|2304582.6699999957|
|587a5f95-1d52-413...| 2301212.690000003|
|9b35272a-84da-4e7...|2263671.6199999973|
|a40d5534-0c30-4da...|2299729.7999999993|
|fd24cab1-ab7f-4dd...|        2315035.42|
+--------------------+------------------+
only showing top 5 rows



# Optimized Joins For Data integration

In [80]:
from pyspark.sql.functions import *

In [81]:
orders_items_joined_df = orders_df.join(order_item_df,'order_id','inner')

In [82]:
orders_items_products_df = orders_items_joined_df.join(products_df,'product_id','inner')

In [83]:
orders_items_products_sellers_df = orders_items_products_df.join(broadcast(sellers_df),'seller_id','inner')

In [84]:
full_orders_df = orders_items_products_sellers_df   # no customers.csv in US E-Commerce, so no customer join

In [85]:
# Geolocation Data (broadcast, joined on seller zip code)

full_orders_df = full_orders_df.join(broadcast(geolocation_unique_df),full_orders_df.seller_zip_code_prefix == geolocation_unique_df.zip_code_prefix,'left')

In [86]:
full_orders_df = full_orders_df.join(reviews_df,'order_id','left')

In [87]:
full_orders_df = full_orders_df.join(payments_df,'order_id','left')

In [88]:
full_orders_df.cache()

26/11/01 20:40:01 WARN CacheManager: Asked to cache already cached data.


DataFrame[order_id: string, seller_id: string, product_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, discount_rate: double, product_category_name: string, product_name: string, product_brand: string, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, cost: double, base_price: double, seller_company_name: string, seller_contact_name: string, seller_contact_gender: string, seller_contact_age: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: int, review_comment_title: string, re

# Aggregation

In [89]:
# Total Orders Per Customer
# Note: in this dataset customer_id is an order-level id (customer_unique_id lived in customers.csv)

customer_order_count_df = full_orders_df.groupBy('customer_id')\
.agg(count('order_id').alias('total_orders'))\
.orderBy(desc('total_orders'))
         
customer_order_count_df.show(5)

+--------------------+------------+
|         customer_id|total_orders|
+--------------------+------------+
|2d3c795e-27d8-4c9...|          12|
|bbbf381b-bf35-414...|          12|
|aca750c5-190c-4ff...|          12|
|1406b4f7-5781-4fc...|          12|
|f9b34ac9-ae32-447...|          12|
+--------------------+------------+
only showing top 5 rows



In [90]:
#Average Review Score Per Seller

seller_review_df = full_orders_df.groupBy('seller_id')\
.agg(avg('review_score').alias('avg_review_score'))\
.orderBy(desc('avg_review_score'))

seller_review_df.show()

+--------------------+------------------+
|           seller_id|  avg_review_score|
+--------------------+------------------+
|47b0f76a-d78e-461...| 4.053023850085179|
|93d73919-4417-44a...| 4.049087749782798|
|b848f631-e120-442...| 4.037843551797041|
|7b9eb34c-3e18-4cf...|  4.03725782414307|
|9a4649de-c63d-430...| 4.035363046662482|
|787bea37-010c-4f7...|4.0349935594675825|
|824a0bea-6c10-417...|  4.03369190176578|
|3a840d5a-70f3-45c...| 4.031878476679504|
|dbaf73c9-ca00-4fe...| 4.030186255619782|
|e4914a38-917e-460...| 4.030121768852809|
|30836545-dfd2-4b3...| 4.027538726333907|
|785471a4-3d30-4ec...| 4.025250417362271|
|c946b185-742f-405...| 4.023819561551433|
|2a3069f9-a45f-448...| 4.023294509151414|
|026fcbbf-9a50-49b...| 4.021734498188792|
|d13e40f6-bf9d-4d5...| 4.021611414183802|
|658762cc-8e95-4e0...| 4.020365315977325|
|0f095623-ea53-49c...| 4.018863879957127|
|2a14b04d-b132-435...| 4.018824513700062|
|de07b19e-30d5-4dd...| 4.018653516295026|
+--------------------+------------

In [91]:
# Top 10 Most Sold Products

top_products_df = full_orders_df.groupBy('product_id')\
.agg(count('order_id').alias('total_sold'))\
.orderBy(desc('total_sold'))\
.limit(10)

top_products_df.show()

+--------------------+----------+
|          product_id|total_sold|
+--------------------+----------+
|fe4ce716-fbd4-4b9...|      2928|
|415911fe-34d5-405...|      2924|
|20fe7673-f03a-4cf...|      2914|
|549f38cb-4c1b-4af...|      2910|
|2685a2fa-5c61-4bc...|      2904|
|9086b2b2-6100-410...|      2898|
|bd9d6e82-4969-4f6...|      2893|
|7fc0ed8d-6f56-427...|      2891|
|11943eeb-7f7a-41d...|      2889|
|86a2a578-0918-403...|      2887|
+--------------------+----------+



# Window Function and Ranking

In [92]:
from pyspark.sql.window import Window

In [93]:
# Dense Rank for Sellers Based on Revenue

window_spec = Window.partitionBy('seller_id').orderBy(desc('price'))

In [94]:
# Rank Top Selling Products Per seller

top_seller_products_df = full_orders_df.withColumn('rank',rank().over(window_spec)).filter(col('rank')<=5)

top_seller_products_df.select('seller_id','price','rank').show()

+--------------------+-------+----+
|           seller_id|  price|rank|
+--------------------+-------+----+
|00c15904-9b03-427...|1999.85|   1|
|00c15904-9b03-427...|1999.85|   1|
|00c15904-9b03-427...|1999.85|   1|
|00c15904-9b03-427...|1999.85|   1|
|00c15904-9b03-427...|1999.85|   1|
|00c15904-9b03-427...|1999.85|   1|
|01bd49b6-ff12-4f3...|1999.85|   1|
|01bd49b6-ff12-4f3...|1999.85|   1|
|01bd49b6-ff12-4f3...|1999.85|   1|
|01bd49b6-ff12-4f3...|1999.85|   1|
|01bd49b6-ff12-4f3...|1999.85|   1|
|01bd49b6-ff12-4f3...|1999.85|   1|
|01bd49b6-ff12-4f3...|1999.85|   1|
|01bd49b6-ff12-4f3...|1999.85|   1|
|026fcbbf-9a50-49b...|1999.85|   1|
|026fcbbf-9a50-49b...|1999.85|   1|
|026fcbbf-9a50-49b...| 1993.1|   3|
|026fcbbf-9a50-49b...| 1993.1|   3|
|026fcbbf-9a50-49b...| 1993.1|   3|
|026fcbbf-9a50-49b...| 1993.1|   3|
+--------------------+-------+----+
only showing top 20 rows



# Advance Aggregation and Enrichment

In [95]:
full_orders_df

DataFrame[order_id: string, seller_id: string, product_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, discount_rate: double, product_category_name: string, product_name: string, product_brand: string, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, cost: double, base_price: double, seller_company_name: string, seller_contact_name: string, seller_contact_gender: string, seller_contact_age: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: int, review_comment_title: string, re

In [96]:
# Total Revenue & Average Order Value (AOV) per Customer

customer_spending_df = full_orders_df.groupBy('customer_id')\
.agg(
    count('order_id').alias('total_orders'),
    sum('price').alias('total_spent'),
    round(avg('price'),2).alias('AOV')
)\
.orderBy(desc('total_spent'))


customer_spending_df.show()

+--------------------+------------+------------------+-------+
|         customer_id|total_orders|       total_spent|    AOV|
+--------------------+------------+------------------+-------+
|97c9c5ef-41c6-4ca...|          12|           18055.0|1504.58|
|80fb158f-9aec-4b0...|          12|          16257.34|1354.78|
|135d03e3-e591-447...|          10|16087.439999999999|1608.74|
|64947d1a-d7de-476...|          12|15729.899999999998|1310.82|
|e573388e-b0c3-43e...|          10|15061.379999999997|1506.14|
|971f212e-b4a8-43b...|          12|14945.619999999999|1245.47|
|905ba961-6369-46c...|          12|14894.260000000002|1241.19|
|df0fd9c2-5bf3-4e0...|          12|14878.440000000002|1239.87|
|9c18ffa4-11d1-4d5...|           8|           14840.8| 1855.1|
|74e830e7-0400-4b1...|          12|          14762.32|1230.19|
|0257eb30-788b-402...|          12|          14678.06|1223.17|
|9a3e8e55-41ba-4bf...|          10|14652.760000000002|1465.28|
|cd01f979-dc61-420...|          10|          14640.68|1

In [97]:
# Seller Performance Metrics ( Revenue, Average Review, Order Count)

seller_performance_df = full_orders_df.groupBy('seller_id') \
.agg(
    count('order_id').alias('total_orders'),
    sum('price').alias('total_revenue'),
    round(avg('review_score'),2).alias('avg_review_score'),
    round(stddev('price'),2).alias('price_variability')
)\
.orderBy(desc('total_revenue'))

In [98]:
seller_performance_df.show()

+--------------------+------------+------------------+----------------+-----------------+
|           seller_id|total_orders|     total_revenue|avg_review_score|price_variability|
+--------------------+------------+------------------+----------------+-----------------+
|db2a9dd9-bd20-430...|        5211|2458665.8899999997|            3.98|           557.04|
|59b0c0b4-2716-458...|        5038|2417471.6999999997|            3.97|           556.46|
|c0e29887-4950-4a9...|        5128| 2413755.599999996|            3.98|           555.78|
|00c6e54e-1372-4d1...|        5113| 2387539.799999999|            3.96|           552.34|
|5392dcc1-9cae-4cd...|        5183|2385842.0800000015|            3.97|           554.28|
|824a0bea-6c10-417...|        5280|2385793.0900000012|            4.03|           545.22|
|4bc06d3b-da05-4bd...|        5186|2384223.7099999986|            4.01|           549.45|
|f842f711-6bf2-41e...|        5177|        2382577.89|            3.97|            538.2|
|9a32f5b3-

In [99]:
# Product Popularity Metrics

product_metrics_df = full_orders_df.groupBy('product_id')\
.agg(
    count('order_id').alias('total_sales'),
    sum('price').alias('total_revenue'),
    round(avg('price'),2).alias('avg_price'),
    round(stddev('price'),2).alias('price_volatility'),\
    collect_set('seller_id').alias('unique_sellers')    
)\
.orderBy(desc('total_sales'))

In [100]:
product_metrics_df.show()

+--------------------+-----------+------------------+---------+----------------+--------------------+
|          product_id|total_sales|     total_revenue|avg_price|price_volatility|      unique_sellers|
+--------------------+-----------+------------------+---------+----------------+--------------------+
|fe4ce716-fbd4-4b9...|       2928|2714197.4399999697|   926.98|             0.0|[82d7ffe0-6c25-4c...|
|415911fe-34d5-405...|       2924| 4113453.960000049|  1406.79|             0.0|[e2d2ffeb-2223-47...|
|20fe7673-f03a-4cf...|       2914| 3664762.959999992|  1257.64|             0.0|[e2d2ffeb-2223-47...|
|549f38cb-4c1b-4af...|       2910| 4007477.399999985|  1377.14|             0.0|[9049c0a7-0a97-45...|
|2685a2fa-5c61-4bc...|       2904| 3169570.800000004|  1091.45|             0.0|[9049c0a7-0a97-45...|
|9086b2b2-6100-410...|       2898| 5613570.899999989|  1937.05|             0.0|[e2d2ffeb-2223-47...|
|bd9d6e82-4969-4f6...|       2893|         1177740.3|    407.1|             0.0|[e

**Homework – Monthly Revenue and Order Count Trend**

From `order_purchase_timestamp` → month, calculate: total_orders, total_revenue, avg_order_value, min_order_value, max_order_value.

In [101]:
# Customer Retention Analysis ( First & Last Order )

customer_retention_df = full_orders_df.groupBy('customer_id')\
.agg(
    first('order_purchase_timestamp').alias('first_order_date'),
    last('order_purchase_timestamp').alias('last_order_date'),
    count('order_id').alias('total_orders'),
    round(avg('price'),2).alias('aov')
)\
.orderBy(desc('total_orders'))

In [102]:
customer_retention_df.show()

+--------------------+-------------------+-------------------+------------+------+
|         customer_id|   first_order_date|    last_order_date|total_orders|   aov|
+--------------------+-------------------+-------------------+------------+------+
|f40eb31e-bbaa-490...|2025-07-26 15:47:51|2025-07-26 15:47:51|          12|548.89|
|3c0b2795-7cf2-455...|2020-01-17 14:48:54|2020-01-17 14:48:54|          12|124.61|
|289f84a6-a213-4e7...|2025-12-21 12:02:46|2025-12-21 12:02:46|          12|383.94|
|e8d72dff-fe33-4f5...|2022-02-18 21:05:34|2022-02-18 21:05:34|          12|354.39|
|9b905594-c786-4d1...|2025-11-08 21:59:58|2025-11-08 21:59:58|          12|712.65|
|66206d64-dcdc-47c...|2022-12-14 18:28:51|2022-12-14 18:28:51|          12|994.25|
|2745cb6f-d2b5-47a...|2024-01-15 11:22:25|2024-01-15 11:22:25|          12|388.62|
|50940c59-d2c1-4db...|2024-10-05 23:34:58|2024-10-05 23:34:58|          12|785.12|
|20482422-1d34-44b...|2025-04-12 12:15:18|2025-04-12 12:15:18|          12|549.99|
|5eb

# Extended Enrichment 

In [103]:
# Order Status Flags

full_orders_df = full_orders_df.withColumn('is_delivered', when(col('order_status')== 'delivered',lit(1)).otherwise(lit(0)))\
.withColumn('is_canceled', when(col('order_status')== 'canceled',lit(1)).otherwise(lit(0)))

In [104]:
full_orders_df.where(full_orders_df['order_status']=='canceled').select('order_status','is_delivered','is_canceled').show(100)

+------------+------------+-----------+
|order_status|is_delivered|is_canceled|
+------------+------------+-----------+
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|


In [105]:
# Order Revenue Calcualtion

full_orders_df = full_orders_df.withColumn('order_revenue',col('price')+col('freight_value'))

In [106]:
full_orders_df.select('price','freight_value','order_revenue').show()

+-------+-------------+------------------+
|  price|freight_value|     order_revenue|
+-------+-------------+------------------+
| 455.53|       117.18|            572.71|
|1196.01|         5.92|           1201.93|
|1162.83|        93.48|           1256.31|
|1162.83|        93.48|           1256.31|
|  14.49|         62.9|             77.39|
|  23.18|       144.21|167.39000000000001|
| 146.39|       124.32|            270.71|
| 185.43|       178.21|            363.64|
|1591.03|        83.38|1674.4099999999999|
|  48.48|         86.7|            135.18|
|  43.93|        83.49|127.41999999999999|
|  43.93|        83.49|127.41999999999999|
|  147.4|       114.68|262.08000000000004|
|  147.4|       114.68|262.08000000000004|
|  19.48|        119.7|            139.18|
|  19.48|        119.7|            139.18|
|  68.83|        92.63|161.45999999999998|
|  68.83|        92.63|161.45999999999998|
| 122.82|        76.26|199.07999999999998|
| 122.82|        76.26|199.07999999999998|
+-------+--

In [107]:
customer_spending_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- total_spent: double (nullable = true)
 |-- AOV: double (nullable = true)



In [108]:
# Customer Segmentation based on spending

customer_spending_df = customer_spending_df.withColumn(
    'customer_segment',
    when(col('AOV') >=1200,"High-Value")
    .when( (col('AOV')<1200) & (col('AOV') >=700),'Medium_Value')
    .otherwise('Low-Value'))

In [109]:
customer_spending_df.show()

+--------------------+------------+------------------+-------+----------------+
|         customer_id|total_orders|       total_spent|    AOV|customer_segment|
+--------------------+------------+------------------+-------+----------------+
|97c9c5ef-41c6-4ca...|          12|           18055.0|1504.58|      High-Value|
|80fb158f-9aec-4b0...|          12|          16257.34|1354.78|      High-Value|
|135d03e3-e591-447...|          10|16087.439999999999|1608.74|      High-Value|
|64947d1a-d7de-476...|          12|15729.899999999998|1310.82|      High-Value|
|e573388e-b0c3-43e...|          10|15061.379999999997|1506.14|      High-Value|
|971f212e-b4a8-43b...|          12|14945.619999999999|1245.47|      High-Value|
|905ba961-6369-46c...|          12|14894.260000000002|1241.19|      High-Value|
|df0fd9c2-5bf3-4e0...|          12|14878.440000000002|1239.87|      High-Value|
|9c18ffa4-11d1-4d5...|           8|           14840.8| 1855.1|      High-Value|
|74e830e7-0400-4b1...|          12|     

In [110]:
full_orders_df = full_orders_df.join(customer_spending_df.select('customer_id','customer_segment'),'customer_id',how='left')

In [111]:
full_orders_df.select('customer_id','customer_segment').show()

+--------------------+----------------+
|         customer_id|customer_segment|
+--------------------+----------------+
|5a861cbc-b1d0-484...|       Low-Value|
|5a861cbc-b1d0-484...|       Low-Value|
|fc99da5b-8977-4f1...|    Medium_Value|
|fc99da5b-8977-4f1...|    Medium_Value|
|4392f6c4-901b-43e...|    Medium_Value|
|4392f6c4-901b-43e...|    Medium_Value|
|15106704-147b-45d...|       Low-Value|
|15106704-147b-45d...|       Low-Value|
|15106704-147b-45d...|       Low-Value|
|f3de55ff-433e-4dc...|       Low-Value|
|f3de55ff-433e-4dc...|       Low-Value|
|f3de55ff-433e-4dc...|       Low-Value|
|ab5cd1a3-64fd-47f...|       Low-Value|
|ab5cd1a3-64fd-47f...|       Low-Value|
|833e18a8-11eb-4a9...|       Low-Value|
|833e18a8-11eb-4a9...|       Low-Value|
|833e18a8-11eb-4a9...|       Low-Value|
|5c57d3e5-58d4-454...|      High-Value|
|bc5bf0f9-f049-48e...|       Low-Value|
|bc5bf0f9-f049-48e...|       Low-Value|
+--------------------+----------------+
only showing top 20 rows



In [112]:
full_orders_df.select('order_purchase_timestamp').show()

+------------------------+
|order_purchase_timestamp|
+------------------------+
|     2022-01-08 19:39:11|
|     2022-01-08 19:39:11|
|     2024-05-19 09:20:26|
|     2024-05-19 09:20:26|
|     2021-06-07 07:33:17|
|     2021-06-07 07:33:17|
|     2025-08-03 15:19:21|
|     2025-08-03 15:19:21|
|     2025-05-12 19:26:04|
|     2025-05-12 19:26:04|
|     2024-06-16 22:30:43|
|     2024-06-16 22:30:43|
|     2024-06-16 22:30:43|
|     2024-06-16 22:30:43|
|     2024-06-16 22:30:43|
|     2024-06-16 22:30:43|
|     2024-06-16 22:30:43|
|     2024-06-16 22:30:43|
|     2024-06-16 22:30:43|
|     2024-06-16 22:30:43|
+------------------------+
only showing top 20 rows



In [113]:
#Hourly Order Distribution
full_orders_df = full_orders_df.withColumn('hour_of_day',expr('hour(order_purchase_timestamp)'))

In [114]:
full_orders_df.select('order_purchase_timestamp','hour_of_day').show()

+------------------------+-----------+
|order_purchase_timestamp|hour_of_day|
+------------------------+-----------+
|     2022-01-08 19:39:11|         19|
|     2022-01-08 19:39:11|         19|
|     2024-05-19 09:20:26|          9|
|     2024-05-19 09:20:26|          9|
|     2021-06-07 07:33:17|          7|
|     2021-06-07 07:33:17|          7|
|     2025-08-03 15:19:21|         15|
|     2025-08-03 15:19:21|         15|
|     2025-05-12 19:26:04|         19|
|     2025-05-12 19:26:04|         19|
|     2024-06-16 22:30:43|         22|
|     2024-06-16 22:30:43|         22|
|     2024-06-16 22:30:43|         22|
|     2024-06-16 22:30:43|         22|
|     2024-06-16 22:30:43|         22|
|     2024-06-16 22:30:43|         22|
|     2024-06-16 22:30:43|         22|
|     2024-06-16 22:30:43|         22|
|     2024-06-16 22:30:43|         22|
|     2024-06-16 22:30:43|         22|
+------------------------+-----------+
only showing top 20 rows



In [115]:
# Weekday vs Weekend Order

full_orders_df = full_orders_df.withColumn('order_day_type',\
                                           when(dayofweek('order_purchase_timestamp').isin(1,7),lit('Weekend')).otherwise(lit('weekday')))

In [116]:
full_orders_df.select('order_purchase_timestamp','order_day_type').show()

+------------------------+--------------+
|order_purchase_timestamp|order_day_type|
+------------------------+--------------+
|     2022-01-08 19:39:11|       Weekend|
|     2022-01-08 19:39:11|       Weekend|
|     2024-05-19 09:20:26|       Weekend|
|     2024-05-19 09:20:26|       Weekend|
|     2021-06-07 07:33:17|       weekday|
|     2021-06-07 07:33:17|       weekday|
|     2025-08-03 15:19:21|       Weekend|
|     2025-08-03 15:19:21|       Weekend|
|     2025-05-12 19:26:04|       weekday|
|     2025-05-12 19:26:04|       weekday|
|     2024-06-16 22:30:43|       Weekend|
|     2024-06-16 22:30:43|       Weekend|
|     2024-06-16 22:30:43|       Weekend|
|     2024-06-16 22:30:43|       Weekend|
|     2024-06-16 22:30:43|       Weekend|
|     2024-06-16 22:30:43|       Weekend|
|     2024-06-16 22:30:43|       Weekend|
|     2024-06-16 22:30:43|       Weekend|
|     2024-06-16 22:30:43|       Weekend|
|     2024-06-16 22:30:43|       Weekend|
+------------------------+--------

In [117]:
full_orders_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- discount_rate: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_brand: string (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_he

In [118]:
!hadoop fs -mkdir -p /data/us-ecom-processed/

In [119]:
full_orders_df.write.mode('overwrite').parquet('/data/us-ecom-processed/')

In [120]:
!hadoop fs -ls -h /data/us-ecom-processed/

Found 20 items
-rw-r--r--   2 root hadoop          0 2026-11-01 20:43 /data/us-ecom-processed/_SUCCESS
-rw-r--r--   2 root hadoop     16.9 M 2026-11-01 20:43 /data/us-ecom-processed/part-00000-96f7d12d-7c92-4d32-be92-46e45eb32c18-c000.snappy.parquet
-rw-r--r--   2 root hadoop     16.9 M 2026-11-01 20:43 /data/us-ecom-processed/part-00001-96f7d12d-7c92-4d32-be92-46e45eb32c18-c000.snappy.parquet
-rw-r--r--   2 root hadoop     16.9 M 2026-11-01 20:43 /data/us-ecom-processed/part-00002-96f7d12d-7c92-4d32-be92-46e45eb32c18-c000.snappy.parquet
-rw-r--r--   2 root hadoop     16.7 M 2026-11-01 20:43 /data/us-ecom-processed/part-00003-96f7d12d-7c92-4d32-be92-46e45eb32c18-c000.snappy.parquet
-rw-r--r--   2 root hadoop     17.0 M 2026-11-01 20:43 /data/us-ecom-processed/part-00004-96f7d12d-7c92-4d32-be92-46e45eb32c18-c000.snappy.parquet
-rw-r--r--   2 root hadoop     16.7 M 2026-11-01 20:43 /data/us-ecom-processed/part-00005-96f7d12d-7c92-4d32-be92-46e45eb32c18-c000.snappy.parquet
-rw-r--r--   2 

# Module 4 - Performance Optimization

### **Key Considerations for Your Dataproc Cluster**

1.  **Cluster Resources:**

    -   **Master:** `n2-standard-4` (4 vCPUs, 16 GB RAM, 32GB disk)
    -   **Workers (2x):** `n2-standard-4` (4 vCPUs, 16 GB RAM, 64GB disk each)
    -   **Total:** 8 worker vCPUs, ~32 GB RAM (excluding master node)
2.  **Dataproc Features Disabled:**

    -   No **autoscaling**, **Metastore**, **advanced execution layer**, **advanced optimizations**
    -   **Storage:** `pd-balanced` (no SSDs, so I/O optimization is crucial)
    -   **Networking:** Internal IP **enabled**
3.  **Optimization Strategy:**

    -   Tune **shuffle partitions**, **broadcast join threshold**, and **storage persistence**
    -   Adjust **parallelism** based on **2 workers x 4 cores**
    -   Avoid **excessive caching** due to **disk-based storage**

In [121]:
# https://spark.apache.org/docs/latest/configuration.html

In [122]:
from pyspark.sql import SparkSession

In [123]:
spark = SparkSession.builder \
.appName('US Ecommerce Performance Optimization') \
.config('spark.executor.memory','6g') \
.config('spark.executor.cores','4') \
.config('spark.executor.instances','2') \
.config('spark.driver.memory','4g') \
.config('spark.driver.maxResultSize','2g') \
.config('spark.sql.shuffle.partitions','64') \
.config('spark.default.parallelism','64') \
.config('spark.sql.adaptive.enabled','true') \
.config('spark.sql.adaptive.coalescePartition.enabled','true') \
.config('spark.sql.autoBroadcastJoinThreshold',20*1024*1024) \
.config('spark.sql.files.maxPartitionBytes','64MB') \
.config('spark.sql.files.openCostInBytes','2MB') \
.config('spark.memory.fraction',0.8) \
.config('spark.memory.storageFraction',0.2) \
.getOrCreate()

26/11/01 20:43:25 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [124]:
full_orders_df = spark.read.parquet('/data/us-ecom-processed/')

In [125]:
full_orders_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- discount_rate: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_brand: string (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_he

# Optimized Join Stragies

In [126]:
# Broadcast (sellers table is small, so broadcast it)

sellers_broadcast_df = broadcast(sellers_df)
optimized_broadcast_join = full_orders_df.join(sellers_broadcast_df,'seller_id')

In [127]:
# Sort and Merge join

sorted_sellers_df = sellers_df.sortWithinPartitions('seller_id')
sorted_orders_df = full_orders_df.sortWithinPartitions('seller_id')


optimized_merge_full_orders_df = sorted_orders_df.join(sorted_sellers_df,'seller_id')

In [128]:
# Bucket join

bucketed_sellers_df = sellers_df.repartition(10,'seller_id')
bucketed_orders_df = full_orders_df.repartition(10,'seller_id')

bucket_join_df = bucketed_orders_df.join(bucketed_sellers_df,'seller_id')

In [129]:
# Skew Join handling

skew_handled_join = full_orders_df.join(sellers_df,'seller_id')

# Module 5 - Data Serving

### Objective:

- Save and retrieve processed data efficiently inside Dataproc.
- Serve data in a structured way for analysis.
- Use Parquet, Hive, and CSV 

In [130]:
full_orders_df = spark.read.parquet('/data/us-ecom-processed/')

In [131]:
full_orders_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- discount_rate: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_brand: string (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_he

In [132]:
# save as Parquet in hdfs

full_orders_df.write.mode('overwrite').parquet('/data/us-ecom-serving/parquet')

In [133]:
# Save it as a parquet in Google Cloud Storage (replace with your own bucket, then uncomment)

# full_orders_df.write.mode('overwrite').parquet('gs://<your-bucket>/temp_data')

In [134]:
full_orders_df.write.mode('overwrite').saveAsTable('full_order_detail')

In [135]:
spark.sql('show tables').show()

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
|  default| cleaned_products|      false|
|  default|full_order_detail|      false|
+---------+-----------------+-----------+



In [136]:
full_orders_df.write.mode('overwrite').option('header','true').csv('/data/us-ecom-serving/csv/')

## 